# Task 3 — RAG Core Logic & Qualitative Evaluation
**Project:** CrediTrust Complaint RAG Chatbot

This notebook:
1. Loads the full-dataset FAISS index built from `complaint_embeddings.parquet`
2. Wires up the retriever, prompt template, and Hugging Face Inference API generator (`src/`)
3. Runs the pipeline against 8 representative questions
4. Builds the Markdown evaluation table required for the report

**Before running:** create a file named `.env` in the project root (already
gitignored) containing one line: `HF_TOKEN=hf_xxxxxxxxxxxxxxxxxxxxxxxxx` with
your real token. The next cell loads it automatically, regardless of which
kernel/terminal you're running this in (a plain `$env:HF_TOKEN = ...` set in
a PowerShell window is NOT reliably inherited by a Jupyter kernel running in
VS Code, since they may be separate processes -- `.env` avoids that entirely).
Also make sure you've already run `src/build_faiss_index.py` so
`vector_store/full_dataset.faiss` and `vector_store/full_dataset_metadata.parquet`
exist.


In [ ]:
import sys
sys.path.insert(0, "../src")

from dotenv import load_dotenv
load_dotenv("../.env", override=True)  # .env always wins, even over a stale value in this kernel/terminal

import os
import pandas as pd

from vector_index import load_index_and_metadata
from embedding import Embedder
from retriever import Retriever
from generator import Generator
from rag_pipeline import RAGPipeline

pd.set_option("display.max_colwidth", 200)


In [ ]:
INDEX_PATH = "../vector_store/full_dataset.faiss"
METADATA_PATH = "../vector_store/full_dataset_metadata.parquet"

index, metadata_df = load_index_and_metadata(INDEX_PATH, METADATA_PATH)
print(f"Loaded index with {index.ntotal:,} chunks.")


In [ ]:
embedder = Embedder()  # sentence-transformers/all-MiniLM-L6-v2, same as Task 2
retriever = Retriever(index, metadata_df, embedder)

# Requires HF_TOKEN to be set as an environment variable.
generator = Generator(model_name="deepseek-ai/DeepSeek-V3-0324")

pipeline = RAGPipeline(retriever, generator, k=5)
print("Pipeline ready.")


## Test questions

Eight questions spanning all four products plus a couple of cross-product
themes (fraud, customer service) that should surface complaints from
multiple categories at once.

In [ ]:
TEST_QUESTIONS = [
    "Why are people unhappy with Credit Cards?",
    "What are the most common complaints about Personal Loans?",
    "What issues do customers report with Savings Accounts?",
    "What problems are customers experiencing with Money Transfers?",
    "Are there complaints about unauthorized transactions?",
    "Do customers complain about poor customer service?",
    "What billing or fee-related issues appear across products?",
    "Are there any complaints related to fraud?",
]
len(TEST_QUESTIONS)


In [ ]:
def truncate(text, n=160):
    text = str(text).replace("\n", " ").strip()
    return text if len(text) <= n else text[:n].rstrip() + "..."


eval_rows = []
for question in TEST_QUESTIONS:
    result = pipeline.answer(question, k=5)
    top_sources = result["sources"][:2]
    sources_str = " | ".join(
        f"[{s['product_category']}] {truncate(s['chunk_text'], 120)}" for s in top_sources
    )
    eval_rows.append({
        "Question": question,
        "Generated Answer": result["answer"],
        "Retrieved Sources (top 2)": sources_str,
        "Quality Score (1-5)": None,   # fill in manually after reading the answer
        "Comments/Analysis": "",        # fill in manually after reading the answer
    })
    print(f"Done: {question}")

eval_df = pd.DataFrame(eval_rows)
eval_df


## Review each answer

For each row above:
1. Read the generated answer against its retrieved sources — does the answer
   actually reflect what the sources say, or does it overreach/hallucinate?
2. Score 1 (poor) to 5 (excellent) in `Quality Score (1-5)`.
3. Add a short note in `Comments/Analysis` — e.g. *"Answer correctly
   summarizes the two sources, but only retrieved Credit Card chunks even
   though Savings Accounts also have similar complaints"* or *"Hallucinated
   a dollar amount not present in any retrieved source."*

Edit the cell below directly with your scores once you've reviewed the
table above (replace the `None`/empty values).

In [ ]:
# Fill these in after manually reviewing the answers above, in the same
# order as TEST_QUESTIONS.
quality_scores = [None, None, None, None, None, None, None, None]
comments = ["", "", "", "", "", "", "", ""]

eval_df["Quality Score (1-5)"] = quality_scores
eval_df["Comments/Analysis"] = comments
eval_df


## Export the Markdown table for the report

In [ ]:
markdown_table = eval_df.to_markdown(index=False)
print(markdown_table)

with open("../reports/task3_evaluation_table.md", "w") as f:
    f.write("# Task 3 — Qualitative Evaluation\n\n")
    f.write(markdown_table)
    f.write("\n")

print("\nSaved to reports/task3_evaluation_table.md")


## Summary observations

After filling in the scores above, write 2-3 sentences here (for your own
reference, and to paste into the final report) on overall patterns: Did the
retriever consistently pull relevant chunks? Did the LLM stick to the
provided context, or did it occasionally answer from general knowledge?
Were any product categories harder to get good answers for than others?